In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error, r2_score

In [2]:
data = pd.read_csv(r'data/data_preprocessed.csv')
print(data.shape)

(278725, 22)


In [3]:
data_dev, data_test = train_test_split(data, test_size=5000/278725, random_state=42)
data_train, data_val = train_test_split(data_dev, test_size=0.2, random_state=42)

print(data_train.shape)
print(data_val.shape)
print(data_test.shape)

(218980, 22)
(54745, 22)
(5000, 22)


In [4]:
X_train = data_train.drop(columns=['precio_pesos_constantes'])
y_train = data_train['precio_pesos_constantes']
X_val = data_val.drop(columns=['precio_pesos_constantes'])
y_val = data_val['precio_pesos_constantes']
X_test = data_test.drop(columns=['precio_pesos_constantes'])
y_test = data_test['precio_pesos_constantes']

print(X_train.shape)
print(y_train.shape)
print(X_val.shape)
print(y_val.shape)
print(X_test.shape)
print(y_test.shape)

(218980, 21)
(218980,)
(54745, 21)
(54745,)
(5000, 21)
(5000,)


In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.1, max_depth=12, random_state=42, eval_metric='rmse')

In [7]:
model.fit(X_train_scaled, y_train, verbose=True)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=2000, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [8]:
y_train_pred = model.predict(X_train_scaled)
rmse = root_mean_squared_error(y_train, y_train_pred)
print('RMSE train:', rmse)
r2 = r2_score(y_train, y_train_pred)
print('R2 train:', r2)

RMSE train: 246579.34037021492
R2 train: 0.8645225203699924


In [9]:
y_val_pred = model.predict(X_val_scaled)
rmse = root_mean_squared_error(y_val, y_val_pred)
print('RMSE val:', rmse)
r2 = r2_score(y_val, y_val_pred)
print('R2 val:', r2)

RMSE val: 436412.6238486941
R2 val: 0.5670964951502149


In [10]:
param_grid = {
    'n_estimators': [100, 200, 500, 1000],
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'max_depth': [7, 9, 12],
    'subsample': [0.6, 0.8, 1.0],
    'gamma': [0, 1, 5, 10],
    'min_child_weight': [1, 3, 5, 10]
}

grid_search = GridSearchCV(estimator=xgb.XGBRegressor(random_state=42), param_grid=param_grid, cv=2, verbose=2, scoring='neg_root_mean_squared_error')
grid_search.fit(X_train_scaled, y_train)

print(f'Best params: {grid_search.best_params_}')
print(f'Best score: {grid_search.best_score_}')

Fitting 2 folds for each of 2880 candidates, totalling 5760 fits
[CV] END gamma=0, learning_rate=0.01, max_depth=7, min_child_weight=1, n_estimators=100, subsample=0.6; total time=   1.2s
[CV] END gamma=0, learning_rate=0.01, max_depth=7, min_child_weight=1, n_estimators=100, subsample=0.6; total time=   0.7s
[CV] END gamma=0, learning_rate=0.01, max_depth=7, min_child_weight=1, n_estimators=100, subsample=0.8; total time=   0.7s
[CV] END gamma=0, learning_rate=0.01, max_depth=7, min_child_weight=1, n_estimators=100, subsample=0.8; total time=   0.7s
[CV] END gamma=0, learning_rate=0.01, max_depth=7, min_child_weight=1, n_estimators=100, subsample=1.0; total time=   0.5s
[CV] END gamma=0, learning_rate=0.01, max_depth=7, min_child_weight=1, n_estimators=100, subsample=1.0; total time=   0.6s
[CV] END gamma=0, learning_rate=0.01, max_depth=7, min_child_weight=1, n_estimators=200, subsample=0.6; total time=   1.4s


KeyboardInterrupt: 